# Your design, on every page

Build a three-page Common Ground report with reusable Python components, a
styled header, a data table, and finalized page numbers. Edit the values below
and rerun to make the design your own. No HTML strings are needed for authoring.

This notebook is an opt-in **Fullbleed 2.6.0a9** example. The component API is
experimental. All content is fictional. Fullbleed and this notebook are MIT licensed.

Use a standard CPU runtime in Colab or a local Jupyter notebook. Run all cells,
review the page previews, and download your files before leaving the session.

## 1. Install the pinned preview

Fullbleed generates the PDF without a browser or system fonts. The notebook host supplies its display and download interface.

In [ ]:
%pip install --quiet --only-binary=:all: "fullbleed==2.6.0a9"

## 2. Make it yours

Start with the name, headline, accent color, and activity rows. All text values
remain text, including `<` and `&`. After an edit, rerun this cell and every cell
below it. Add activity rows to see the page total change after pagination.

In [ ]:
PROJECT_NAME = "Common Ground"
HEADLINE = "A season of shared space."
ACCENT = "#755122"
ACTIVITY = [
    ("Garden mornings", "12", "84 visits"),
    ("Repair afternoons", "4", "31 items repaired"),
    ("Shared meals", "3", "96 places set"),
    ("Seed exchanges", "2", "47 packets shared"),
    ("Welcome walks", "6", "28 neighbors"),
]

## 3. Give the components a shared style

`Theme` supplies inherited text and named colors. `styled(...)` creates reusable
defaults. `class_name="folio"` below adds a literal class; the `.folio` selector
in `Document(styles=...)` defines how the page counter looks.

In [ ]:
# SPDX-License-Identifier: MIT
"""A fictional seasonal brief built from reusable component values."""
from fullbleed.ui.experimental import (
    Box, Column, DataTable, Document, Flow, Heading, Page, PageBand, PageBreak,
    PageNumbers, Paragraph, Row, Stack, Strong, Theme, component, mm, pt, style,
    styled, token,
)


THEME = Theme(
    {"ink": "#183e34", "muted": "#53665c", "paper": "#edf2e6", "accent": ACCENT},
    text=style(font_family="Inter", font_size=pt(10), line_height=1.45, color=token("ink")),
)
Kicker = styled(Paragraph, font_size=pt(7), font_weight=700, letter_spacing=pt(1),
                text_transform="uppercase", color=token("accent"))
Title = styled(Heading, font_size=pt(31), line_height=1.08, letter_spacing=pt(-0.8))
Small = styled(Paragraph, font_size=pt(8), color=token("muted"))

## 4. Compose reusable pieces

`Brand`, `Metric`, and `NextStep` return ordinary component trees. Named `part`
values let a caller use `.with_parts(...)` to customize an internal element.
The same `Brand()` value can be used in a page band or in document content.

In [ ]:
@component(parts=("name", "detail"))
def Brand():
    return Stack(
        Paragraph(PROJECT_NAME.upper(), part="name", style=style(font_size=pt(8), font_weight=700, letter_spacing=pt(0.8))),
        Paragraph("A neighborhood, growing together.", part="detail", style=style(font_size=pt(6), color=token("muted"))),
        gap=pt(2),
    )


@component(parts=("value", "label"))
def Metric(value, label):
    return Stack(
        Paragraph(value, part="value", style=style(font_size=pt(32), font_weight=700, line_height=1.1)),
        Small(label, part="label"), gap=pt(5),
    )


@component(parts=("heading", "body"))
def NextStep(number, title, text):
    return Stack(
        Kicker(number),
        Heading(title, level=3, part="heading", style=style(font_size=pt(16))),
        Paragraph(text, part="body"), gap=pt(6),
    )

## 5. Put the report together

`PageBand` places components in left, center, or right margin slots.
`PageNumbers(total=True)` uses final page counts. Page margins must leave enough
space for the bands; a fit error identifies the page, slot, and authored source.

The bands repeat as PDF artifacts outside body reading order. Essential identity
and report information also belong in the body.

The short lesson box uses `break_inside="avoid"` to keep its heading and body
together when a longer activity table pushes it to the next page.

In [ ]:
def build_document():
    activity = ACTIVITY
    return Document(
        Flow(
            Kicker("Field notes / Autumn 2026"),
            Title(HEADLINE, level=1),
            Paragraph("A small record of what happens when neighbors have a place to meet, make, and lend a hand."),
            Box(
                Row(Metric("126", "neighbors took part"), Metric("21", "open sessions"), gap=pt(18)),
                style=style(background=token("paper"), padding=pt(18)),
            ),
            Heading("The place is the invitation", level=2, style=style(font_size=pt(17))),
            Paragraph("We kept the gate open, the table long, and the activities simple. New faces became familiar ones. Practical work made room for conversation."),
            Small(f"{PROJECT_NAME} is a fictional community project. Every figure in this brief is sample data."),
            gap=pt(15),
        ),
        PageBreak(),
        Flow(
            Kicker("01 / What we made possible"),
            Title("Small actions. Shared results.", level=2),
            Paragraph("The busiest sessions mixed a useful task with time to stay. No specialist knowledge was needed to join in."),
            DataTable(
                activity, caption="Season activity", columns=(
                    Column("Activity", lambda row: row[0]),
                    Column("Sessions", lambda row: row[1], align="right", width=mm(18)),
                    Column("Participation", lambda row: row[2], width=mm(37)),
                ),
            ).with_parts(
                caption=style(font_size=pt(9), font_weight=700, margin_bottom=pt(7)),
                header=style(background=token("paper"), font_size=pt(8), color=token("ink")),
                cell=style(font_size=pt(8), padding=pt(8)),
            ),
            Box(
                Kicker("A useful lesson"),
                Paragraph(Strong("Make the first step visible. "), "A welcome at the gate, a clearly marked tool bench, and a task that takes ten minutes helped people begin."),
                style=style(background=token("paper"), padding=pt(15), break_inside="avoid"),
            ),
            Small("Visits and participation counts can include the same neighbor more than once. They describe activity, not unique attendance."),
            gap=pt(12),
        ),
        PageBreak(),
        Flow(
            Kicker("02 / What comes next"),
            Title("Keep the welcome growing.", level=2),
            Paragraph("Three modest commitments will guide the next season. Each has a clear owner and a way for someone new to help."),
            NextStep("01", "Make room at the table", "Reserve a few places at every shared meal for neighbors joining for the first time. Publish the menu and access information together."),
            NextStep("02", "Pass on a practical skill", "Pair each repair session with a short demonstration. Keep a simple printed guide beside the tools for people who prefer to follow at their own pace."),
            NextStep("03", "Listen before adding more", "Ask what made participation easy and what got in the way. Use those answers to choose the next opening hours and activities."),
            Small(f"Prepared by {PROJECT_NAME} / October 2026. This sample is authored with Fullbleed components, including its repeated header and finalized page numbers."),
            gap=pt(15),
        ),
        title=f"{PROJECT_NAME} — Field notes", lang="en-US", theme=THEME,
        page=Page(
            "A5", margin=(mm(20), mm(17), mm(20), mm(17)),
            header=PageBand(left=Brand(), right=Kicker("Field notes / 03").with_style(font_size=pt(6))),
            footer=PageBand(
                left=Small("AUTUMN 2026 · SAMPLE DATA").with_style(font_size=pt(6)),
                right=PageNumbers(total=True, prefix="", separator=" / ", class_name="folio"),
            ),
        ),
        styles={".folio": style(font_size=pt(8), color=token("muted"))},
    )


document = build_document()

## 6. Render PDF and export HTML

The bundled Inter font is registered explicitly for PDF generation. Styled HTML
keeps its font-family declaration; a browser host supplies Inter or a fallback.
Unstyled HTML keeps content and classes while removing generated and inline styles.
Both exports contain one header/main/footer sequence and explicit `[page]` /
`[pages]` placeholders. They do not add JavaScript or browser pagination.

The PDF/UA-2 profile and source checks express authoring intent. Review and
validate your own output; this notebook does not establish arbitrary-document
accessibility or replace human assistive-technology testing.

In [ ]:
from importlib import metadata, resources
from pathlib import Path
from fullbleed.ui.experimental import Renderer

assert metadata.version("fullbleed") == "2.6.0a9"
output = Path("output")
output.mkdir(exist_ok=True)
font = resources.files("fullbleed_assets").joinpath("fonts/Inter-Variable.ttf")
renderer = Renderer(
    font_files=[str(font)], pdf_profile="pdfua2",
    document_timestamp="2026-10-19T00:00:00Z",
)
prepared = document.prepare(renderer, a11y_mode="raise")
pdf_path = output / "field-notes.pdf"
pdf_path.write_bytes(prepared.render_pdf())
for include_styles, filename in (
    (True, "field-notes.styled.html"),
    (False, "field-notes.unstyled.html"),
):
    (output / filename).write_text(
        document.to_html(include_styles=include_styles, a11y_mode="raise"),
        encoding="utf-8", newline="\n",
    )
print(f"Saved {pdf_path} and both HTML exports.")

In [ ]:
from IPython.display import Image, display

preview_paths = []
for number, png in enumerate(prepared.render_image_pages(dpi=110), start=1):
    path = output / f"field-notes-page-{number}.png"
    path.write_bytes(png)
    preview_paths.append(path)
    display(Image(data=png, width=540))
print(f"{len(preview_paths)} pages. Review every page before using the document.")

## 7. Download the current result

The ZIP contains the PDF, both HTML exports, and this run's page previews. In
Colab the next cell starts a download; local Jupyter shows a download link.
The individual files also remain in `output/`. Shorter reruns exclude old,
longer-run previews from the ZIP.

In [ ]:
import sys
from zipfile import ZipFile, ZIP_DEFLATED

bundle_path = output / "component-field-notes.zip"
current_files = [pdf_path, output / "field-notes.styled.html",
                 output / "field-notes.unstyled.html", *preview_paths]
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for path in current_files:
        bundle.write(path, arcname=path.name)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(bundle_path))
else:
    from IPython.display import FileLink
    display(FileLink(bundle_path.as_posix()))

## Keep composing

- [Page-band guide and complete source kit](https://docs.fullbleed.dev/guides/component-preview-9/)
- [API, fit diagnostics, and limitations](https://github.com/fullbleed-engine/fullbleed-official/blob/v2.6.0a9/docs/component-page-bands.md)
- [Northstar and Common Ground through both authoring paths](https://docs.fullbleed.dev/guides/showcase-authoring/)
- [Share a small authoring example](https://github.com/fullbleed-engine/fullbleed-official/discussions/86)

Native streaming output is not atomic. Managed fit checks are conservative.
Named or facing-page band variants are not part of this preview. Bootstrap
components and HTML exports remain static.